# Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from pathlib import Path
import numpy as np
import math
from sklearn.utils.class_weight import compute_class_weight

# Модель

In [ ]:
class CrossAttentionWithChunks(nn.Module):
    def __init__(self, text_dim=312, audio_dim=768, hidden_dim=128, num_chunks_text=8, num_chunks_audio=12):
        super().__init__()

        # Размеры чанков
        self.text_chunk_size = text_dim // num_chunks_text
        self.audio_chunk_size = audio_dim // num_chunks_audio

        # Проекции чанков в hidden_dim
        self.text_proj = nn.Linear(self.text_chunk_size, hidden_dim)
        self.audio_proj = nn.Linear(self.audio_chunk_size, hidden_dim)

        # Оригинальные проекции для Q, K, V
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.scale = math.sqrt(hidden_dim)

        self.num_chunks_text = num_chunks_text
        self.num_chunks_audio = num_chunks_audio

    def forward(self, text_emb, audio_emb):
        # text_emb: [batch, 312]
        # audio_emb: [batch, 768]

        # 1. Разбиваем на чанки
        text_chunks = self._chunk_embedding(text_emb, self.text_chunk_size, self.num_chunks_text)
        audio_chunks = self._chunk_embedding(audio_emb, self.audio_chunk_size, self.num_chunks_audio)

        # 2. Проекция чанков в hidden_dim
        text_seq = self.text_proj(text_chunks)
        audio_seq = self.audio_proj(audio_chunks)

        # 3. Проекции в Q, K, V
        Q = self.query_proj(text_seq)      # [batch, 6, hidden]
        K = self.key_proj(audio_seq)       # [batch, 8, hidden]
        V = self.value_proj(audio_seq)     # [batch, 8, hidden]

        # 4. Attention: Q × K^T
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [batch, 6, 8]
        attn_weights = torch.softmax(scores, dim=-1)               # [batch, 6, 8]

        # 5. Взвешенная сумма
        attended = torch.matmul(attn_weights, V)  # [batch, 6, hidden]

        # 6. Пулинг по seq_len (берём среднее)
        attended_pooled = attended.mean(dim=1)    # [batch, hidden]

        # 7. Выход
        output = self.out_proj(attended_pooled)   # [batch, hidden]

        return output

    def _chunk_embedding(self, emb, chunk_size, num_chunks):
        """emb: [batch, dim] -> [batch, num_chunks, chunk_size]"""
        batch_size, dim = emb.shape
        # Обрезаем до кратного chunk_size
        emb = emb[:, :num_chunks * chunk_size]
        return emb.view(batch_size, num_chunks, chunk_size)

class MultimodalCrossAttentionClassifier(nn.Module):
    def __init__(self, text_dim=312, audio_dim=768, hidden_dim=128,
                 num_chunks_text=8, num_chunks_audio=12, num_classes=3, dropout=0.7):
        super().__init__()

        self.cross_attention = CrossAttentionWithChunks(
            text_dim, audio_dim, hidden_dim,
            num_chunks_text, num_chunks_audio
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, audio_emb, text_emb):
        attended = self.cross_attention(text_emb, audio_emb)
        logits = self.classifier(attended)
        return logits

# Эмбеддинги

In [ ]:
def load_embeddings(file_path):
    """Загружает эмбеддинги из файла {имя: эмбеддинг}"""
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                name, emb_str = parts
                emb = np.array([float(x) for x in emb_str.split(',')])
                embeddings[name] = emb
    return embeddings

# Пути к файлам (пути к Google Диску)
audio_emb_path = "/content/drive/MyDrive/876_augmented/Emo_Emb.txt"
text_emb_path = "/content/drive/MyDrive/876_augmented/RuBert_Emb.txt"

print("Загрузка эмбеддингов...")
audio_embs = load_embeddings(audio_emb_path)
text_embs = load_embeddings(text_emb_path)
print(f"Аудио эмбеддингов: {len(audio_embs)}")
print(f"Текст эмбеддингов: {len(text_embs)}")

# Находим общие имена
common_names = set(audio_embs.keys()) & set(text_embs.keys())
print(f"Общих примеров: {len(common_names)}")

Загрузка эмбеддингов...
Аудио эмбеддингов: 3504
Текст эмбеддингов: 3504
Общих примеров: 3504


In [ ]:
situation_list = ['threat', 'warning', 'neutral']
situation2id = {s: i for i, s in enumerate(situation_list)}
id2situation = {i: s for s, i in situation2id.items()}

X_audio = []
X_text = []
y = []

for name in common_names:
    # Обстановка — третья часть имени файла
    parts = name.split('_')
    if len(parts) >= 3:
        situation = parts[2]  # 'threat', 'warning', 'neutral'
        if situation in situation2id:
            X_audio.append(audio_embs[name])
            X_text.append(text_embs[name])
            y.append(situation2id[situation])
        else:
            print(f"⚠️ Неизвестная обстановка: {situation} в {name}")
    else:
        print(f"⚠️ Не хватает частей в имени: {name}")

print(f"Загружено {len(y)} примеров")
print(f"Распределение: {np.unique(y, return_counts=True)}")


⚠️ Неизвестная обстановка: 0019 в absent_absent_0019
⚠️ Неизвестная обстановка: 0023 в absent_absent_0023_aug1
⚠️ Неизвестная обстановка: 0045 в absent_absent_0045_aug0
⚠️ Неизвестная обстановка: 0018 в absent_absent_0018_aug2
⚠️ Неизвестная обстановка: 0009 в absent_absent_0009_aug2
⚠️ Неизвестная обстановка: 0027 в absent_absent_0027_aug0
⚠️ Неизвестная обстановка: 0007 в absent_absent_0007_aug0
⚠️ Неизвестная обстановка: 0032 в absent_absent_0032_aug1
⚠️ Неизвестная обстановка: 0040 в absent_absent_0040_aug2
⚠️ Неизвестная обстановка: 0021 в absent_absent_0021_aug2
⚠️ Неизвестная обстановка: 0014 в absent_absent_0014
⚠️ Неизвестная обстановка: 0008 в absent_absent_0008
⚠️ Неизвестная обстановка: 0016 в absent_absent_0016_aug2
⚠️ Неизвестная обстановка: 0043 в absent_absent_0043
⚠️ Неизвестная обстановка: 0008 в absent_absent_0008_aug0
⚠️ Неизвестная обстановка: 0026 в absent_absent_0026_aug2
⚠️ Неизвестная обстановка: 0028 в absent_absent_0028_aug0
⚠️ Неизвестная обстановка: 0031 в 

In [ ]:
X_audio_train, X_audio_temp, X_text_train, X_text_temp, y_train, y_temp = train_test_split(
    X_audio, X_text, y, test_size=0.4, random_state=42, stratify=y
)

X_audio_val, X_audio_test, X_text_val, X_text_test, y_val, y_test = train_test_split(
    X_audio_temp, X_text_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

Train: 1989, Val: 663, Test: 664


In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, audio_embs, text_embs, labels):
        self.audio_embs = [torch.tensor(emb, dtype=torch.float) for emb in audio_embs]
        self.text_embs = [torch.tensor(emb, dtype=torch.float) for emb in text_embs]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.audio_embs[idx], self.text_embs[idx], self.labels[idx]

train_dataset = MultimodalDataset(X_audio_train, X_text_train, y_train)
val_dataset = MultimodalDataset(X_audio_val, X_text_val, y_val)
test_dataset = MultimodalDataset(X_audio_test, X_text_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Обучение

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalCrossAttentionClassifier(num_classes=3)
model.to(device)

'''class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)'''
class_weights = torch.tensor([4.0, 3.0, 1.0]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
# Scheduler для уменьшения lr при застое
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # смотрим на val loss
    patience=3,       # ждём 3 эпохи без улучшения
    factor=0.5        # уменьшаем lr в 2 раза
)

print(f"Модель на {device}")
print(f"Параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Модель на cuda
Параметров: 83715


In [ ]:
best_val_loss = float('inf')
best_model_state = None

for epoch in range(100):
    # Train
    model.train()
    train_loss = 0
    for audio_emb, text_emb, labels in train_loader:
        audio_emb = audio_emb.to(device)
        text_emb = text_emb.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(audio_emb, text_emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for audio_emb, text_emb, labels in val_loader:
            audio_emb = audio_emb.to(device)
            text_emb = text_emb.to(device)
            labels = labels.to(device)

            logits = model(audio_emb, text_emb)
            loss = criterion(logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_true, val_preds)

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()
        print(f"✅ Лучшая модель (Val Loss: {best_val_loss:.4f}, Val Acc: {val_acc:.4f})")

    print(f"Эпоха {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

✅ Лучшая модель (Val Loss: 0.6945, Val Acc: 0.7104)
Эпоха  1 | Train Loss: 0.6802 | Val Loss: 0.6945 | Val Acc: 0.7104
✅ Лучшая модель (Val Loss: 0.6945, Val Acc: 0.7104)
Эпоха  2 | Train Loss: 0.6627 | Val Loss: 0.6945 | Val Acc: 0.7104
✅ Лучшая модель (Val Loss: 0.6945, Val Acc: 0.7104)
Эпоха  3 | Train Loss: 0.6909 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  4 | Train Loss: 0.6919 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  5 | Train Loss: 0.6930 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  6 | Train Loss: 0.6850 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  7 | Train Loss: 0.6725 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  8 | Train Loss: 0.6741 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха  9 | Train Loss: 0.6764 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха 10 | Train Loss: 0.6648 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха 11 | Train Loss: 0.6789 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха 12 | Train Loss: 0.6682 | Val Loss: 0.6945 | Val Acc: 0.7104
Эпоха 13 | Train Loss: 0.6798 | Val Loss

In [ ]:
model.load_state_dict(best_model_state)
model_path = "/content/drive/MyDrive/multimodal_crossattention_full.pth"
torch.save(model, model_path)
print(f"✅ Полная модель сохранена в {model_path}")

# Также сохраняем только веса (для подстраховки)
weights_path = "/content/drive/MyDrive/multimodal_crossattention_weights.pth"
torch.save(best_model_state, weights_path)
print(f"✅ Веса сохранены в {weights_path}")

# Тестирование

In [ ]:
model.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for audio_emb, text_emb, labels in test_loader:
        audio_emb = audio_emb.to(device)
        text_emb = text_emb.to(device)
        labels = labels.to(device)

        logits = model(audio_emb, text_emb)
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_true, test_preds)
print(f"\n{'='*50}")
print(f"Тест Accuracy: {test_acc:.4f}")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=situation_list))


Тест Accuracy: 0.6747

Classification Report:
              precision    recall  f1-score   support

      threat       0.68      0.76      0.72       164
     warning       0.42      0.54      0.47       167
     neutral       0.87      0.70      0.77       333

    accuracy                           0.67       664
   macro avg       0.66      0.67      0.66       664
weighted avg       0.71      0.67      0.69       664



# Загрузка модели

In [ ]:
# Вариант А: загружаем полную модель (проще)
model = torch.load("/content/drive/MyDrive/multimodal_crossattention_full.pth")
model.eval()

# Вариант Б: загружаем веса (нужно сначала создать модель)
model = MultimodalCrossAttentionClassifier(num_classes=3)
model.load_state_dict(torch.load("/content/drive/MyDrive/multimodal_crossattention_weights.pth"))
model.eval()